# RT June Fixes Runner (Vast / multi-GPU)

Runs every GPU fix from the June review in one resumable pass, then the CPU
reanalyses and manuscript assets.

Workflow:
1. configure and clone the repo
2. install dependencies into `.venv`
3. prepare public benchmark JSONLs (MSC valid, LongMemEval-S cleaned)
4. launch `june_fixes/multigpu/run_june_fixes_multigpu.sh` (queued, per-GPU
   workers, resumable — re-run the cell after any interruption)
5. CPU post-processing: answer-harm Gate 1, multiple-comparison sweep,
   regime detector, manuscript assets
6. zip everything under `results/june_fixes/` for download


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/SteveMama/RT.git"  # or upload a zip and point REPO_ZIP at it
REPO_DIR = Path("/workspace/RT").resolve()
HF_TOKEN = ""           # required only for the gated llama32_3b cross-family probe
MODEL_KEY = "qwen25_15b"
BUDGETS = "0.20,0.35,0.50"
GPU_COUNT = 0            # 0 = use all visible GPUs
EXTRACT_BATCH_SIZE = 8   # 8 for T4/L4, 16 for A100/H100
EXTRACT_CACHE_ROOT = str(REPO_DIR / "results" / "paper3" / "extract_cache")
RUN_BASELINES = "1"
RUN_QA = "1"
RUN_CROSSFAMILY = "1"
RUN_LME_SCALEUP = "0"    # set "1" for the 40-conversation / 80-turn LongMemEval rerun


In [ ]:
import subprocess, sys

def run(cmd, cwd=None, env=None):
    print("$", cmd, flush=True)
    process = subprocess.Popen(
        cmd, shell=True, cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"command failed ({process.returncode}): {cmd}")


In [ ]:
# Clone + environment
if not REPO_DIR.exists():
    run(f"git clone {REPO_URL} {REPO_DIR}")
run("python3 -m venv .venv", cwd=REPO_DIR)
run(".venv/bin/pip install -q --upgrade pip", cwd=REPO_DIR)
run(".venv/bin/pip install -q -e . huggingface_hub tqdm pillow matplotlib", cwd=REPO_DIR)
# Optional published-compressor baseline (fix 3); harmless if it fails.
run(".venv/bin/pip install -q llmlingua || true", cwd=REPO_DIR)
run(".venv/bin/python -c \"import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'gpus')\"", cwd=REPO_DIR)
run("nvidia-smi")


In [ ]:
# Optional HF login (needed for llama32_3b)
if HF_TOKEN:
    run(f".venv/bin/python -c \"from huggingface_hub import login; login('{HF_TOKEN}')\"", cwd=REPO_DIR)


In [ ]:
# Prepare public benchmark JSONLs (skipped if already present)
BENCH = REPO_DIR / "benchmarks"
MSC_JSONL = BENCH / "msc_valid_normalized.jsonl"
LME_JSONL = BENCH / "longmemeval_s_cleaned_normalized.jsonl"
if not MSC_JSONL.exists():
    run(".venv/bin/python scripts/download_public_benchmark.py --benchmark msc_valid", cwd=REPO_DIR)
    run(f".venv/bin/python scripts/prepare_public_benchmark_jsonl.py --format msc --output-path {MSC_JSONL}", cwd=REPO_DIR)
if not LME_JSONL.exists():
    run(".venv/bin/python scripts/download_public_benchmark.py --benchmark longmemeval_s", cwd=REPO_DIR)
    run(f".venv/bin/python scripts/prepare_public_benchmark_jsonl.py --format longmemeval --output-path {LME_JSONL}", cwd=REPO_DIR)
print("MSC:", MSC_JSONL.exists(), "| LME:", LME_JSONL.exists())


In [ ]:
# Main multi-GPU run (resumable: just re-run this cell after an interruption)
env = {
    **os.environ,
    "GPU_COUNT": str(GPU_COUNT),
    "EXTRACT_BATCH_SIZE": str(EXTRACT_BATCH_SIZE),
    "EXTRACT_CACHE_ROOT": EXTRACT_CACHE_ROOT,
    "MODEL_KEY": MODEL_KEY,
    "BUDGETS": BUDGETS,
    "MSC_INPUT": str(MSC_JSONL),
    "LONGMEM_INPUT": str(LME_JSONL),
    "RUN_BASELINES": RUN_BASELINES,
    "RUN_QA": RUN_QA,
    "RUN_CROSSFAMILY": RUN_CROSSFAMILY,
}
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN
run("bash june_fixes/multigpu/run_june_fixes_multigpu.sh", cwd=REPO_DIR, env=env)


In [ ]:
# Optional: LongMemEval scale-up (fix 6) — 40 conversations, 80-turn cap
if RUN_LME_SCALEUP == "1":
    env_lme = {**env}
    run(f"bash june_fixes/longmemeval_scaleup/run_lme_scaleup.sh {MSC_JSONL} {LME_JSONL} {MODEL_KEY}",
        cwd=REPO_DIR, env=env_lme)


In [ ]:
# CPU post-processing: answer-harm Gate 1 (fix 2) on the merged oracle rows
ORACLE_DIRS = {
    "msc_valid": REPO_DIR / "paper3_gate1_scaleup_multigpu_merged_results" / "paper3_gate1_scaleup_multigpu_oracle_msc_valid_32conv",
    "longmemeval_s_cleaned": REPO_DIR / "paper3_gate1_scaleup_multigpu_merged_results" / "paper3_gate1_scaleup_multigpu_oracle_longmemeval_s_cleaned_12conv",
}
for bench, oracle_dir in ORACLE_DIRS.items():
    rows = oracle_dir / "candidate_rows.csv"
    if rows.exists():
        run(f".venv/bin/python -m june_fixes.answer_harm_oracle.answer_harm_gate1 "
            f"--candidate-rows {rows} --benchmark-name {bench} "
            f"--output-dir results/june_fixes/answer_harm_oracle/{bench}", cwd=REPO_DIR)
    else:
        print("missing", rows)


In [ ]:
# CPU post-processing: multiple-comparison sweep (fix 4) + regime detector (fix 7)
run(".venv/bin/python -m june_fixes.stats.multiple_comparisons "
    "--search-roots results,artifacts,paper3_gate1_scaleup_multigpu_merged_results,paper3 "
    "--output-dir results/june_fixes/multiple_comparisons", cwd=REPO_DIR)

hardset_rows = REPO_DIR / "results" / "paper3" / "harm_oracle" / "paper3_oracle_hardset_smoke" / "candidate_rows.csv"
msc_rows = ORACLE_DIRS["msc_valid"] / "candidate_rows.csv"
lme_rows = ORACLE_DIRS["longmemeval_s_cleaned"] / "candidate_rows.csv"
if all(path.exists() for path in (hardset_rows, msc_rows, lme_rows)):
    run(f".venv/bin/python -m june_fixes.regime_detector.regime_detector "
        f"--labeled-csv hardset={hardset_rows} --labeled-csv msc={msc_rows} "
        f"--labeled-csv longmemeval={lme_rows} "
        f"--output-dir results/june_fixes/regime_detector", cwd=REPO_DIR)
else:
    print("regime detector skipped; missing one of", hardset_rows, msc_rows, lme_rows)


In [ ]:
# Manuscript assets (fix 8)
run(".venv/bin/python june_fixes/manuscript/implementation_details_appendix.py", cwd=REPO_DIR)
run(".venv/bin/python june_fixes/manuscript/regime_map_figure.py", cwd=REPO_DIR)


In [ ]:
# Inspect key reports inline
from pathlib import Path
for report in sorted(Path(REPO_DIR, "results", "june_fixes").rglob("*report.md")):
    print("\n" + "=" * 100)
    print(report)
    print("=" * 100)
    print(report.read_text()[:4000])


In [ ]:
# Package everything for download
import shutil
archive = shutil.make_archive(str(REPO_DIR / "june_fixes_results"), "zip",
                              root_dir=str(REPO_DIR / "results"), base_dir="june_fixes")
print("download:", archive)
